If you check the code please check the last cell first please.

Part 0

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from gensim.models import Word2Vec
from sklearn.model_selection import ParameterGrid
import re
import warnings
warnings.filterwarnings("ignore") # ignoring all warnings that the code is faced

# Preprocessing the text by converting to lowercase (My very simple preprocessing)
def preprocess(text):
    return re.findall(r'\b\w+\b', text.lower())

def load_data(file_path):
    return pd.read_csv(file_path)

def extract_text_column(df):
    col = df.columns[0] 
    return df[col].astype(str).tolist() # Converting to a list of strings

# Generating n-grams (sequences of 'n' consecutive tokens) from our tokens
def custom_ngrams(tokens, n):
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

Part 1

In [2]:
def generate_ngrams(corpus, n):
    sequences = []
    for sentence in corpus:
        tokens = preprocess(sentence)
        for gram in custom_ngrams(tokens, n):
            sequences.append(gram)
    return sequences

def train_word2vec(corpus):
    tokenized = [preprocess(sent) for sent in corpus]
    model = Word2Vec(sentences=tokenized, vector_size=100, window=5, min_count=1, workers=4)
    return model

# Creating input and target tensors for training
def create_dataset(ngrams_list, w2v_model):
    inputs, targets = [], []
    for gram in ngrams_list:
        if all(token in w2v_model.wv for token in gram):
            input_vec = np.concatenate([w2v_model.wv[token] for token in gram[:-1]])
            target_vec = w2v_model.wv[gram[-1]]
            inputs.append(input_vec)
            targets.append(target_vec)
    return torch.tensor(inputs), torch.tensor(targets)

Part 2

In [3]:
class FeedForwardNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(FeedForwardNN, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim) # Fully connected layer
        self.relu = nn.ReLU() # ReLU activation
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        return self.fc2(x)

# Train the neural network model using MSE loss and Adam optimizer
def train_model(model, train_X, train_y, lr=0.01, epochs=10):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        output = model(train_X)
        loss = criterion(output, train_y)
        loss.backward()
        optimizer.step()
    return model

def evaluate_perplexity(model, test_X, test_y):
    model.eval()
    with torch.no_grad():
        predictions = model(test_X)
        loss = nn.MSELoss()(predictions, test_y)
        return torch.exp(loss).item()

Part 3

In [4]:
def run_pipeline(train_text, valid_text, test_text):
    w2v = train_word2vec(train_text)
    best_perplexity = float('inf')
    best_config = None
    results = {}
    for n in [2, 3]:
        train_ngrams = generate_ngrams(train_text, n)
        valid_ngrams = generate_ngrams(valid_text, n)
        test_ngrams = generate_ngrams(test_text, n)
        train_X, train_y = create_dataset(train_ngrams, w2v)
        valid_X, valid_y = create_dataset(valid_ngrams, w2v)
        test_X, test_y = create_dataset(test_ngrams, w2v)

        lr_list = [0.001, 0.01, 0.1]
        for lr in lr_list:
            model = FeedForwardNN(train_X.shape[1], 128, train_y.shape[1])
            model = train_model(model, train_X.float(), train_y.float(), lr=lr, epochs=20)
            perp = evaluate_perplexity(model, valid_X.float(), valid_y.float())
            print(f"n={n}, LR={lr}, Validation Perplexity={perp:.2f}")
            if perp < best_perplexity:
                best_perplexity = perp
                best_config = (n, lr)

        best_n, best_lr = best_config
        model = FeedForwardNN(train_X.shape[1], 128, train_y.shape[1])
        model = train_model(model, train_X.float(), train_y.float(), lr=best_lr, epochs=20)
        test_perp = evaluate_perplexity(model, test_X.float(), test_y.float())
        results[n] = test_perp

    print("\nTest Perplexities:")
    for n in results:
        print(f"{n}-gram FFNN Perplexity: {results[n]:.2f}")
    best_n = min(results, key=results.get)
    print(f"Best FFNN Model: {best_n}-gram")

def main():
    train_df = load_data("E:/Aidin/Universities/Amirkabir (Master)/Lessons/My-Terms/Term 3/Natural Language Processing/Assignments/HW1/Question/Dataset/train.csv")
    valid_df = load_data("E:/Aidin/Universities/Amirkabir (Master)/Lessons/My-Terms/Term 3/Natural Language Processing/Assignments/HW1/Question/Dataset/val.csv")
    test_df = load_data("E:/Aidin/Universities/Amirkabir (Master)/Lessons/My-Terms/Term 3/Natural Language Processing/Assignments/HW1/Question/Dataset/test.csv")

    train_text = extract_text_column(train_df)
    valid_text = extract_text_column(valid_df)
    test_text = extract_text_column(test_df)

    run_pipeline(train_text, valid_text, test_text)

if __name__ == "__main__":
    main()

n=2, LR=0.001, Validation Perplexity=2.58
n=2, LR=0.01, Validation Perplexity=2.35
n=2, LR=0.1, Validation Perplexity=3.07
n=3, LR=0.001, Validation Perplexity=2.52
n=3, LR=0.01, Validation Perplexity=2.33
n=3, LR=0.1, Validation Perplexity=3.36

Test Perplexities:
2-gram FFNN Perplexity: 2.37
3-gram FFNN Perplexity: 2.33
Best FFNN Model: 3-gram


New Version

In [1]:
# Part 0
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from gensim.models import Word2Vec
from sklearn.model_selection import ParameterGrid
import re
import warnings
warnings.filterwarnings("ignore") # ignoring all warnings that the code is faced

# Preprocessing the text by converting to lowercase, removing non-alphanumeric characters & handling numbers as a placeholder word
def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)# Removing non-alphanumeric characters
    text = re.sub(r"\s+", " ", text).strip() # Replacing multiple or all white spaces with a single space
    words = re.findall(r'\b\w+\b', text)
    # Replacing any number (integer or with decimal) with the placeholder "numbers"
    for index, w in enumerate(words):
        if w.replace('.', '', 1).isdigit() or re.search(r'\d', w):
            words[index] = "numbers"
    return words

def load_data(file_path):
    return pd.read_csv(file_path)

def extract_text_column(df):
    col = df.columns[0]
    return df[col].astype(str).tolist() # Converting to a list of strings

# Generating n-grams (sequences of 'n' consecutive tokens) from our tokens
def custom_ngrams(tokens, n):
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

# Part 1 
def generate_ngrams(corpus, n):
    sequences = []
    for sentence in corpus:
        tokens = preprocess(sentence)
        # Skipping sentences that are shorter than n due to the limitation of model
        if len(tokens) < n:  
            continue
        for gram in custom_ngrams(tokens, n):
            sequences.append(gram)
    return sequences

def train_word2vec(corpus):
    tokenized = [preprocess(sent) for sent in corpus if preprocess(sent)] 
    if not tokenized:
        raise ValueError("Empty corpus after preprocessing")
    model = Word2Vec(sentences=tokenized, vector_size=100, window=5, min_count=1, workers=4)
    return model

# Creating input and target tensors for training
def create_dataset(ngrams_list, w2v_model):
    inputs, targets = [], []
    for gram in ngrams_list:
        # Checking that all tokens are in Word2Vec model
        if all(token in w2v_model.wv for token in gram):
            input_vec = np.concatenate([w2v_model.wv[token] for token in gram[:-1]]) # Concatenating vectors for input
            target_vec = w2v_model.wv[gram[-1]]  # Using the last token's vector as target
            inputs.append(input_vec)
            targets.append(target_vec)
    if not inputs:
        return None, None 
    return torch.tensor(inputs, dtype=torch.float32), torch.tensor(targets, dtype=torch.float32)

# Part 2
class FeedForwardNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(FeedForwardNN, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim) # Fully connected layer
        self.relu = nn.ReLU() # ReLU activation
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        return self.fc2(x)

# Train the neural network model using MSE loss and Adam optimizer
def train_model(model, train_X, train_y, lr=0.01, epochs=10):
    if train_X is None or train_y is None:
        return None 
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        output = model(train_X)
        loss = criterion(output, train_y)
        # Checking that loss is NaN or inf
        if torch.isnan(loss) or torch.isinf(loss):
            return None 
        loss.backward()
        optimizer.step()
    return model

def evaluate_perplexity(model, test_X, test_y):
    # Checking for returning infinity if data or model is missing
    if model is None or test_X is None or test_y is None:
        return float('inf')
    model.eval()
    with torch.no_grad():
        predictions = model(test_X)
        loss = nn.MSELoss()(predictions, test_y)
        if torch.isnan(loss) or torch.isinf(loss):
            return float('inf')
        return torch.exp(loss.clamp(min=-10, max=10)).item() 

# Part 3
def run_pipeline(train_text, valid_text, test_text):
    w2v = train_word2vec(train_text)
    best_perplexity = float('inf')
    best_config = None
    results = {}
    param_grid = {
        'n': [2, 3],  # n-gram sizes
        'lr': [0.001, 0.01, 0.1],  # Learning rates
        'hidden_dim': [64, 128, 256],  # Hidden layer dimensions
        'epochs': [10, 20, 30]  # Number of training epochs (Consider my limitations)
    }

    # Iterating over all hyperparameter combinations from the grid
    for params in ParameterGrid(param_grid):
        n = params['n']
        lr = params['lr']
        hidden_dim = params['hidden_dim']
        epochs = params['epochs']
        train_ngrams = generate_ngrams(train_text, n)
        valid_ngrams = generate_ngrams(valid_text, n)
        test_ngrams = generate_ngrams(test_text, n)
        train_X, train_y = create_dataset(train_ngrams, w2v)
        valid_X, valid_y = create_dataset(valid_ngrams, w2v)
        test_X, test_y = create_dataset(test_ngrams, w2v)
        if train_X is None or valid_X is None:
            print(f"n={n}, LR={lr}, HiddenDim={hidden_dim}, Epochs={epochs}: Skipped (empty dataset)")
            continue
        # Getting the number of input features
        input_dim = train_X.shape[1]
        # Getting the number of output features
        output_dim = train_y.shape[1]
        model = FeedForwardNN(input_dim, hidden_dim, output_dim)
        model = train_model(model, train_X, train_y, lr=lr, epochs=epochs)
        if model is None:
            print(f"n={n}, LR={lr}, HiddenDim={hidden_dim}, Epochs={epochs}: Skipped (training failed)")
            continue
        perp = evaluate_perplexity(model, valid_X, valid_y)
        print(f"n={n}, LR={lr}, HiddenDim={hidden_dim}, Epochs={epochs}, Validation Perplexity={perp:.2f}")
        if perp < best_perplexity:
            best_perplexity = perp
            best_config = params
    if best_config is None:
        print("No valid configurations found.")
        return
    n = best_config['n']
    lr = best_config['lr']
    hidden_dim = best_config['hidden_dim']
    epochs = best_config['epochs']

    train_ngrams = generate_ngrams(train_text, n)
    test_ngrams = generate_ngrams(test_text, n)
    train_X, train_y = create_dataset(train_ngrams, w2v)
    test_X, test_y = create_dataset(test_ngrams, w2v)

    if train_X is None or test_X is None:
        print("Best configuration failed on test set (empty dataset).")
        return

    input_dim = train_X.shape[1]
    output_dim = train_y.shape[1]
    model = FeedForwardNN(input_dim, hidden_dim, output_dim)
    model = train_model(model, train_X, train_y, lr=lr, epochs=epochs)
    
    if model is None:
        print("Best configuration failed to train on test set.")
        return

    test_perp = evaluate_perplexity(model, test_X, test_y)
    results[n] = test_perp

    print("\nTest Perplexities:")
    print(f"{n}-gram FFNN Perplexity: {results[n]:.2f}")
    print(f"Best FFNN Model: {n}-gram (LR={lr}, HiddenDim={hidden_dim}, Epochs={epochs})")

# Part 4
def main():
    train_df = load_data("E:/Aidin/Universities/Amirkabir (Master)/Lessons/My-Terms/Term 3/Natural Language Processing/Assignments/HW1/Question/Dataset/train.csv")
    valid_df = load_data("E:/Aidin/Universities/Amirkabir (Master)/Lessons/My-Terms/Term 3/Natural Language Processing/Assignments/HW1/Question/Dataset/val.csv")
    test_df = load_data("E:/Aidin/Universities/Amirkabir (Master)/Lessons/My-Terms/Term 3/Natural Language Processing/Assignments/HW1/Question/Dataset/test.csv")

    train_text = extract_text_column(train_df)
    valid_text = extract_text_column(valid_df)
    test_text = extract_text_column(test_df)

    run_pipeline(train_text, valid_text, test_text)

if __name__ == "__main__":
    main()

n=2, LR=0.001, HiddenDim=64, Epochs=10, Validation Perplexity=2.76
n=3, LR=0.001, HiddenDim=64, Epochs=10, Validation Perplexity=2.63
n=2, LR=0.01, HiddenDim=64, Epochs=10, Validation Perplexity=2.31
n=3, LR=0.01, HiddenDim=64, Epochs=10, Validation Perplexity=2.27
n=2, LR=0.1, HiddenDim=64, Epochs=10, Validation Perplexity=3.35
n=3, LR=0.1, HiddenDim=64, Epochs=10, Validation Perplexity=3.76
n=2, LR=0.001, HiddenDim=128, Epochs=10, Validation Perplexity=2.47
n=3, LR=0.001, HiddenDim=128, Epochs=10, Validation Perplexity=2.41
n=2, LR=0.01, HiddenDim=128, Epochs=10, Validation Perplexity=2.27
n=3, LR=0.01, HiddenDim=128, Epochs=10, Validation Perplexity=2.24
n=2, LR=0.1, HiddenDim=128, Epochs=10, Validation Perplexity=6.75
n=3, LR=0.1, HiddenDim=128, Epochs=10, Validation Perplexity=101.44
n=2, LR=0.001, HiddenDim=256, Epochs=10, Validation Perplexity=2.40
n=3, LR=0.001, HiddenDim=256, Epochs=10, Validation Perplexity=2.36
n=2, LR=0.01, HiddenDim=256, Epochs=10, Validation Perplexity=2.